# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '/product_catalog.csv'#../06-midterm-walmart/data/product_catalog.csv'

try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))
filtered = cat['product_name'].str.lower().str.contains('pop')
pop_rows = cat[filtered]

print(f"Rows matching 'pop': {len(pop_rows)}")
pop_rows

Rows matching 'pop': 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
filtered_strawberry = cat['product_name'].str.lower().str.contains('pop', 'strawberry')
strawberry = cat[filtered_strawberry]

print(f"Rows matching 'pop' and 'strawberry': {len(strawberry)}")

n = cat[~filtered_strawberry]
print("Not strawberry pop tarts:", len(n))


Rows matching 'pop' and 'strawberry': 4
Not strawberry pop tarts: 41


**Pop-Tart rows that are a different product:** 0

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
n_rows_before = len(cat)

CANONICAL = 'SKU#459812'
sku_map = {sku: CANONICAL for sku in strawberry['sku']}
print(sku_map)

cat['canonical_sku'] = cat['sku'].map(sku_map).fillna(cat['sku'])
#cat[['sku', 'canonical_sku', 'product_name']]

{'PT-12': 'SKU#459812', 'SKU#459812': 'SKU#459812', 'POPTART-STRAW': 'SKU#459812', 'PT_STRAWBERRY': 'SKU#459812'}


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
print("Before:")
print(cat['category'].value_counts())

cat['category_clean'] = cat['category'].str.strip().str.title()

print("After:")
print(cat['category_clean'].value_counts())

Before:
category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64
After:
category_clean
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
Health                 2
Breakfast 

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
import re

def name_key(name):
    s = re.sub(r'[^a-z0-9\s]', ' ', str(name).lower())
    return ' '.join(sorted(s.split()))

cat['name_key'] = cat['product_name'].fillna('').map(name_key)

groups = (cat.groupby('name_key')
             .agg(n_skus=('canonical_sku', 'nunique'), skus=('canonical_sku', lambda x: sorted(x.unique())), names=('product_name', lambda x: sorted(x.unique())))
             .query('n_skus > 1')
             .sort_values('n_skus', ascending=False))

groups

,n_skus,skus,names
name_key,,,
board game monopoly,3,"[BG-MONO, GAME-MONO, MONOPOLY-01]","[Board Game - Monopoly, Monopoly Board Game]"
24 beer light pack,2,"[BEER-24, LTBEER24]",[Light Beer (24-pack)]
4 aa batteries pack,2,"[BAT-AA4, BATT_AA_4PK]",[AA Batteries 4-Pack]
aid first kit,2,"[FAID-01, FIRSTAID-KIT]","[First Aid Kit, First-Aid Kit]"
bread white,2,"[BREAD-WH, WHITEBREAD]",[WHITE BREAD]
butter peanut,2,"[PB-16, PNUTBTR-16]","[PEANUT BUTTER, Peanut Butter]"
candles emergency,2,"[CANDLE-EMRG, CNDL-EM]","[Candles (Emergency), Emergency Candles]"
canned soup,2,"[CNSOUP-01, SOUP-CAN]","[Canned Soup, Soup - Canned]"
cards playing,2,"[CARDS-01, PLAYCARD-STD]","[PLAYING CARDS, Playing Cards]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
assert cat['canonical_sku'].notna().all()
assert len(cat) == n_rows_before

unmapped = ~cat['sku'].isin(sku_map.keys())
assert (cat.loc[unmapped, 'canonical_sku'] == cat.loc[unmapped, 'sku']).all()

print(f"OK: {len(cat)} rows, 0 null canonical SKUs, {cat['sku'].nunique()} -> {cat['canonical_sku'].nunique()} distinct SKUs")

OK: 45 rows, 0 null canonical SKUs, 45 -> 42 distinct SKUs


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) Two product families beyond pop tarts that appear fragmented are the monopoly board game, and peanut butter. I would do the same as pop tarts, and normalize each name (lowercase, strip punctuation, sort the words) and group on that key. Then I would review each group myself to confirm it’s one product, then map every variant to a single canonical SKU. Finally, I'd rerun the grouping to confirm nothing is left and the Q6 asserts to confirm no rows were lost or nulled.

b) Consolidation becomes a business decision once the question changes from “are these the same string?” to “do we want to treat these as the same product?” All these decisions are just data cleaning, but differences that could affect pricing, supply, or reporting belong to whoever owns the product.

One examplle is Light Beer (24-pack), BEER-24 vs. LTBEER24. The names match exactly, but “Light Beer” is generic, and these could be two different brands, or a national brand and a store brand, that were entered with the same description. If so, merging them would combine sales, costs, and supplier relationships that need to stay separate.
I’d ask the beverage category manager or buyer, who knows whether these are the same item from the same supplier. This issue probably shouldn't be decided in data cleaning.